<a href="https://colab.research.google.com/github/iamanantalok/Master-Thesis-DES-LCA/blob/main/learning-logs/03_LCA_and_Brightway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Brightway needs four building blocks:**
1. Project
2. Activity
3. Exchange
4. Method

**Activities sit in databases, and there are two kinds:**
*  The technosphere is the economy: things people make (ex- electricity,glass,bottles).
*  The biosphere is nature: what goes into or comes out of the environment (co2 into the air).

**There are three exchange types:**
1. production: what the activity makes (usually 1 of itself)
2. technosphere: something it takes from the economy (the bottle takes electricity)
3. biosphere: something it puts into nature (glassmaking emits CO₂)

In [ ]:
!pip install brightway2

In [ ]:
import brightway2 as bw

In [ ]:
bw.projects.set_current("bottle_line_learning")

#1. The biosphere: things that go into nature
bio = bw.Database("my_biosphere")
bio.write({
    ("my_biosphere","co2"):{"name":"Carbon dioxide","unit":"kilogram","type":"emission"},

})


#2. The technosphere: things we make and buy
tech = bw.Database("my_factory")
tech.write({
    ("my_factory", "electricity"): {
        "name": "electricity, grid",
        "unit": "kilowatt hour",
        "exchanges": [
            {"input": ("my_factory", "electricity"), "amount": 1, "type": "production"},
            {"input": ("my_biosphere", "co2"), "amount": 0.2, "type": "biosphere"},
        ],
    },
    ("my_factory", "glass"): {
        "name": "glass bottle, empty",
        "unit": "unit",
        "exchanges": [
            {"input": ("my_factory", "glass"), "amount": 1, "type": "production"},
            {"input": ("my_biosphere", "co2"), "amount": 0.3, "type": "biosphere"},
        ],
    },
    ("my_factory", "filled_bottle"): {
        "name": "filled and capped bottle",
        "unit": "unit",
        "exchanges": [
            {"input": ("my_factory", "filled_bottle"), "amount": 1, "type": "production"},
            {"input": ("my_factory", "glass"), "amount": 1, "type": "technosphere"},
            {"input": ("my_factory", "electricity"), "amount": 0.13, "type": "technosphere"},
        ],
    },
})

# 3. The method: how much each emission counts for climate change
method_name = ("my method", "climate change", "GWP100")
method = bw.Method(method_name)
method.register(unit="kg CO2-eq")
method.write([(("my_biosphere", "co2"), 1.0)])


# 4. Calculate
bottle = tech.get("filled_bottle")
lca = bw.LCA({bottle: 1}, method_name)
lca.lci()
lca.lcia()
print("Climate impact of 1 bottle:", round(lca.score, 3), "kg CO2-eq")

Writing activities to SQLite3 database:
0% [#] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
Writing activities to SQLite3 database:


Title: Writing activities to SQLite3 database:
  Started: 10/19/2026 10:45:04
  Finished: 10/19/2026 10:45:04
  Total time elapsed: 00:00:00
  CPU %: 0.00
  Memory %: 3.20


0% [###] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 10/19/2026 10:45:04
  Finished: 10/19/2026 10:45:04
  Total time elapsed: 00:00:00
  CPU %: 74.40
  Memory %: 3.20
Climate impact of 1 bottle: 0.326 kg CO2-eq


In [ ]:
for exc in bottle.exchanges():
    print(exc["type"], exc.input["name"], exc["amount"])

production filled and capped bottle 1
technosphere glass bottle, empty 1
technosphere electricity, grid 0.13


#### Machines and States:
        filled bottle
        ├── 1 empty glass bottle
        ├── 1 × filler operation
        │     ├── 0.1177 kWh while processing
        │     ├── 0.0254 kWh while idle
        │     ├── 0.0043 kWh while blocked
        │     └── 0.0011 kWh while down
        └── 1 × capper operation
              ├── 0.1159 kWh while processing
              ├── 0.0107 kWh while idle
              ├── 0.0000 kWh while blocked
              └── 0.0016 kWh while down



    These numbers are from our SimPy run.

In [ ]:
import brightway2 as bw

bw.projects.set_current("bottle_line_v2")


# Biosphere: nature
bw.Database("my_biosphere").write({
    ("my_biosphere", "co2"): {"name": "Carbon dioxide", "unit": "kilogram", "type": "emission"},
})

# Electricity used per bottle, by machine and state (from SimPy stage 5, seed 42)
KWH_PER_BOTTLE = {
    "filler": {"processing": 0.1177, "idle": 0.0254, "blocked": 0.0043, "down": 0.0011},
    "capper": {"processing": 0.1159, "idle": 0.0107, "blocked": 0.0000, "down": 0.0016},
}

def machine_activity(name):
    exchanges = [{"input": ("my_factory", name), "amount": 1, "type": "production"}]
    for state, kwh in KWH_PER_BOTTLE[name].items():
        exchanges.append({
            "input": ("my_factory", "electricity"),
            "amount": kwh,
            "type": "technosphere",
            "state": state,                       # our own label, Brightway ignores it in the maths
        })
    return {"name": f"{name} operation, per bottle", "unit": "unit", "exchanges": exchanges}


bw.Database("my_factory").write({
    # background
    ("my_factory", "electricity"): {
        "name": "electricity, grid", "unit": "kilowatt hour",
        "exchanges": [
            {"input": ("my_factory", "electricity"), "amount": 1, "type": "production"},
            {"input": ("my_biosphere", "co2"), "amount": 0.2, "type": "biosphere"},
        ],
    },
    ("my_factory", "glass"): {
        "name": "glass bottle, empty", "unit": "unit",
        "exchanges": [
            {"input": ("my_factory", "glass"), "amount": 1, "type": "production"},
            {"input": ("my_biosphere", "co2"), "amount": 0.3, "type": "biosphere"},
        ],
    },
    # foreground: one activity per machine
    ("my_factory", "filler"): machine_activity("filler"),
    ("my_factory", "capper"): machine_activity("capper"),
    # the product
    ("my_factory", "filled_bottle"): {
        "name": "filled and capped bottle", "unit": "unit",
        "exchanges": [
            {"input": ("my_factory", "filled_bottle"), "amount": 1, "type": "production"},
            {"input": ("my_factory", "glass"), "amount": 1, "type": "technosphere"},
            {"input": ("my_factory", "filler"), "amount": 1, "type": "technosphere"},
            {"input": ("my_factory", "capper"), "amount": 1, "type": "technosphere"},
        ],
    },
})

method_name = ("my method", "climate change", "GWP100")
m = bw.Method(method_name)
m.register(unit="kg CO2-eq")
m.write([(("my_biosphere", "co2"), 1.0)])

tech = bw.Database("my_factory")
bottle = tech.get("filled_bottle")
lca = bw.LCA({bottle: 1}, method_name)
lca.lci()
lca.lcia()
print("Total per bottle:", round(lca.score, 4), "kg CO2-eq")


Writing activities to SQLite3 database:
0% [#] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
Writing activities to SQLite3 database:


Title: Writing activities to SQLite3 database:
  Started: 10/19/2026 11:02:39
  Finished: 10/19/2026 11:02:39
  Total time elapsed: 00:00:00
  CPU %: 0.00
  Memory %: 3.18


0% [#####] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 10/19/2026 11:02:39
  Finished: 10/19/2026 11:02:39
  Total time elapsed: 00:00:00
  CPU %: 0.00
  Memory %: 3.18
Total per bottle: 0.3553 kg CO2-eq


In [ ]:
def score_of(activity, amount):
    sub = bw.LCA({activity: amount}, method_name)
    sub.lci()
    sub.lcia()
    return sub.score

print("Where does the footprint come from?")
for exc in bottle.technosphere():
    s = score_of(exc.input, exc["amount"])
    print(f"  {exc.input['name']:<30} {s:.4f} kg CO2-eq  ({s / lca.score:.0%})")

Where does the footprint come from?
  glass bottle, empty            0.3000 kg CO2-eq  (84%)
  filler operation, per bottle   0.0297 kg CO2-eq  (8%)
  capper operation, per bottle   0.0256 kg CO2-eq  (7%)


In [ ]:
rows = []
for machine_name in ["filler", "capper"]:
    machine = tech.get(machine_name)
    for exc in machine.technosphere():
        s = score_of(exc.input, exc["amount"])
        rows.append((machine_name, exc["state"], s))

print(f"{'machine':<8} {'state':<11} kg CO2-eq")
for machine_name, state, s in rows:
    print(f"{machine_name:<8} {state:<11} {s:.4f}")

machine  state       kg CO2-eq
filler   processing  0.0235
filler   idle        0.0051
filler   blocked     0.0009
filler   down        0.0002
capper   processing  0.0232
capper   idle        0.0021
capper   blocked     0.0000
capper   down        0.0003
